<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/JARVIS_ML_TrainingGPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Following example shows developing high-quality JARVIS-ML model development


In [ ]:
!git clone --recursive https://github.com/Microsoft/LightGBM
!pip install jarvis-tools
!cd LightGBM && rm -rf build && mkdir build && cd build && cmake -DUSE_GPU=1 ../../LightGBM && make -j4 && cd ../python-package && python3 setup.py install --precompile --gpu;


In [ ]:
!nvidia-smi

In [ ]:
from jarvis.ai.pkgs.lgbm.regression import parameters_dict
from scipy.stats import median_absolute_deviation as mad
from jarvis.ai.pkgs.utils import get_ml_data
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import train_test_split
from jarvis.ai.pkgs.utils import regr_scores

params = parameters_dict()

print (params)

In [ ]:
%%time
mem=[]
for i,j in params.items():
    X,y,jid=get_ml_data(dataset="cfid_3d", ml_property=i)
    lgbm = lgb.LGBMRegressor(device= 'gpu',n_estimators= j['n_estimators'],learning_rate= j['learning_rate'],num_leaves= j['num_leaves'])
    if 'eps' in i:#fit refractive index, not dielectric constant
      y=np.sqrt(y)
    X_train, X_test, y_train, y_test, jid_train, jid_test = train_test_split(X, y, jid, random_state=1, test_size=.1)
    %%timeit lgbm.fit(X_train,y_train)
    pred = lgbm.predict(X_test)
    reg_sc = regr_scores(y_test, pred)
    mae=reg_sc['mae'] #mean absolute error
    madev=mad(y) #mean absolute deviation
    #mae_over_madev=float(mae)/float(madev)
    mem.append([i,len(X),mae,madev])
    print ('Property,Length, MAE,MAD', i,len(X),mae,madev)
    print ()
    print ()
    

In [ ]:
mem_2=[]
for i,j in params.items():
  
    X,y,jid=get_ml_data(dataset="cfid_3d", ml_property=i)
    lgbm = lgb.LGBMRegressor(device= 'gpu',n_estimators= 2*j['n_estimators'],learning_rate= j['learning_rate']/2,num_leaves= 2*j['num_leaves'])
    X_train, X_test, y_train, y_test, jid_train, jid_test = train_test_split(X, y, jid, random_state=1, test_size=.1)
    %time lgbm.fit(X_train,y_train)
    pred = lgbm.predict(X_test)
    reg_sc = regr_scores(y_test, pred)
    mae=reg_sc['mae'] #mean absolute error
    madev=mad(y) #mean absolute deviation
    #mae_over_madev=float(mae)/float(madev)
    mem_2.append([i,len(X),mae,madev])
    print ('Property,Length, MAE,MAD', i,len(X),mae,madev)
    print ()
    print ()
    